# Balkan Statistical Office - Business Analytics Project

## Setup and Imports
In this cell, I imported all the libraries we'll need.
- `pandas` for structuring the data
- `sqlalchemy` ORM handling for database read/writes
- `pyaxis` to parse data returned from the Statistical Databases API
- `dotenv` to load environment variables like the database URL
- `Config` is our own class to handle the JSON config files


In [1]:
import json
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from pyaxis import pyaxis
from sqlalchemy import create_engine, MetaData, Table, select, insert, delete
from sqlalchemy.orm import Session

from Config import Config

### Load environment variables and setup DB connection
Here I load the `.env` file to get the database URL, and then create a SQLAlchemy engine.
This engine will be used later to read/write data from/to the database.


In [2]:
load_dotenv()
engine = create_engine(os.getenv("DB_URL"))

### Load configuration files
This part goes through all JSON files in the `configs/mkstat` folder.
Each JSON file contains settings for a dataset (like which columns we want, how to map values, etc.).
We read them and store them as `Config` objects in a list.


In [3]:
config_files:Path = Path("configs/mkstat")
configs = []

for json_file in config_files.glob("*.json"):
    print("Loading:", json_file.name)
    with open(json_file, "r", encoding="utf-8") as f:
        data = json.load(f)
        configs.append(Config(json_file.name, data))

print(f"Loaded {len(configs)} JSON files.")

Loading: crime.json
Loading: gdp-by-expenditure.json
Loading: gdp-by-production.json
Loading: gdp-by-region.json
Loading: migration.json
Loading: unemployment-rates.json
Loaded 6 JSON files.


### Preview data for a selected config
I pick one of the loaded configs (`configs[1]`) to check what data it points to.
Using `pyaxis.parse()`, we fetch the data and take a quick look at the first few rows.
This is just to make sure everything loads correctly.


In [4]:
currentConfig = configs[1]
print(currentConfig.fileName)
pyaxis.parse(currentConfig.url, currentConfig.encoding)["DATA"].head()

gdp-by-expenditure.json
Multilingual PX file


,Категории,Мерки,Година,DATA
0,Извоз на стоки,тековни цени (мил. денари),2000,45479.0000000
1,Извоз на стоки,тековни цени (мил. денари),2001,41161.0000000
2,Извоз на стоки,тековни цени (мил. денари),2002,38211.0000000
3,Извоз на стоки,тековни цени (мил. денари),2003,41176.0000000
4,Извоз на стоки,тековни цени (мил. денари),2004,51544.0000000


## Function to create a DataFrame from a config
`createDf(config)` takes a `Config` object and turns the data into a pandas DataFrame.
- It selects only the columns defined in the config
- Default values are applied if no field name is present
- Maps values if there's a mapping defined
- Converts the main "DATA" column to numeric
- Drops rows where the value is missing


In [5]:
def createDf(config:Config) -> pd.DataFrame:
    px = pyaxis.parse(config.url, encoding=config.encoding)
    dataDict = {}
    for column in config.columns:
        if column.fieldName is not None:
            series = px["DATA"][column.fieldName]
        else:
            series = column.defaultValue
        if column.map is not None:
            series = pd.Series(series).replace(column.map)
        dataDict[column.columnName] = series
    dataDict["value"] = pd.to_numeric(px["DATA"]["DATA"], errors="coerce")
    return pd.DataFrame(dataDict).dropna(subset=["value"])
df = createDf(currentConfig)
df.head()

Multilingual PX file


,country,approach,type,year,value
0,Republic of North Macedonia,expenditure,Exports of goods,2000,45479.0
1,Republic of North Macedonia,expenditure,Exports of goods,2001,41161.0
2,Republic of North Macedonia,expenditure,Exports of goods,2002,38211.0
3,Republic of North Macedonia,expenditure,Exports of goods,2003,41176.0
4,Republic of North Macedonia,expenditure,Exports of goods,2004,51544.0


## Prepare the DataFrame for the database
- `getOrCreate()` checks if a value already exists in a DB table, and creates it if not. Returns the ID.
- `prepareForDb()` converts all the categorical columns in the DataFrame to their DB IDs using `getOrCreate()`.
- Also adjusts the `value` column if an exchange rate is specified.

After this step, the DataFrame is ready to be inserted into the DB.


In [6]:
def getOrCreate(session: Session, table_name: str, field_name: str, field_value) -> int:
    metadata = MetaData()
    table = Table(table_name, metadata, autoload_with=session.bind)

    stmt = select(table.c.id).where(table.c[field_name] == field_value)
    result = session.execute(stmt).scalar_one_or_none()
    if result:
        return result

    ins = insert(table).values(**{field_name: field_value})
    result = session.execute(ins)
    session.commit()

    return result.inserted_primary_key[0]

def prepareForDb(df:pd.DataFrame, config:Config) -> pd.DataFrame:
    with Session(engine) as session:
        for column in config.columns:
            uniqueValues = df[column.columnName].unique()
            mapping = {}
            for value in uniqueValues:
                valueId = getOrCreate(session, column.tableName, column.columnName, value)
                mapping[value] = valueId
            df[column.columnName] = df[column.columnName].map(mapping).astype(int)
    if config.exchangeRate is not None:
        df["value"] = df["value"].astype(float).apply(lambda x: x / config.exchangeRate)
    return df
df = prepareForDb(df, currentConfig)
df.head()

,country,approach,type,year,value
0,1,1,1,1,739.015275
1,1,1,1,2,668.849529
2,1,1,1,3,620.913227
3,1,1,1,4,669.093273
4,1,1,1,5,837.569061


## Insert DataFrame into the database
- `insertDfIntoDb()` takes a DataFrame and a table name and inserts all rows into the DB.
- Finally, we call it with our prepared DataFrame and the table name from the config.
- After this, the data from the Statistical Database is now in our local database.


In [7]:
def insertDfIntoDb(tableName:str, df: pd.DataFrame):
    country_ids = df["country"].unique().tolist()
    with Session(engine) as session:
        metadata = MetaData()
        table = Table(tableName, metadata, autoload_with=session.bind)
        try:
            session.begin()
            session.execute(delete(table).where(table.c.country.in_(country_ids)))
            rows = df.to_dict(orient="records")
            session.execute(insert(table), rows)
            session.commit()
        except Exception as e:
            session.rollback()
            print("Error inserting data:", e)
            raise

In [10]:
insertDfIntoDb(currentConfig.tableName, df)